# Huggingface pipeline

Install all required packages for the fine-tuning experiment.

In [1]:
import importlib.metadata
import subprocess
import sys

def install_dep(modules: list):
    for pkg in modules:
        base = pkg.split("[")[0]  # strip extras like [torch]
        try:
            importlib.metadata.distribution(base)
            print(f"{pkg} already installed")
        except importlib.metadata.PackageNotFoundError:
            print(f"Installing {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

modules = ["huggingface_hub",
           "datasets", 
           "transformers", 
           "torch", 
           "numpy", 
           "nvidia-ml-py", 
           "transformers[torch]", 
           "evaluate",
           "ipywidgets",
           "scikit-learn",
           "scipy",
           "accelerate"]
install_dep(modules)

huggingface_hub already installed
datasets already installed
transformers already installed
torch already installed
numpy already installed
nvidia-ml-py already installed
transformers[torch] already installed
evaluate already installed
ipywidgets already installed
scikit-learn already installed
scipy already installed
accelerate already installed


Login with notebook to Huggingface

In [2]:
from huggingface_hub import notebook_login
notebook_login()

Load the MRPC dataset from GLUE — 3 668 training and 408 validation sentence pairs labeled as paraphrase or not.

In [3]:
from datasets import load_dataset

raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

mrpc/train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

mrpc/validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

mrpc/test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

Load the BERT tokenizer, define a tokenisation function for sentence pairs, and create a `DataCollatorWithPadding` for dynamic batching.

In [5]:
from transformers import AutoTokenizer, DataCollatorWithPadding
checkpoint = "test-trainer/checkpoint-1377"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenizer_function(item):
    return tokenizer(item["sentence1"],item["sentence2"],truncation=True,padding=True,return_tensors="pt")
data_collactor = DataCollatorWithPadding(tokenizer=tokenizer)

Apply tokenisation to train and validation sets; drop raw text columns, rename `label` → `labels`, and format as PyTorch tensors.

In [6]:
train_dataset = raw_datasets["train"]
train_dataset = train_dataset.map(tokenizer_function,batched=True)
train_dataset = train_dataset.remove_columns(["sentence1","sentence2","idx"])
train_dataset = train_dataset.rename_column("label","labels")
train_dataset.set_format("torch")

validation_dataset = raw_datasets["validation"]
validation_dataset = validation_dataset.map(tokenizer_function,batched=True)
validation_dataset = validation_dataset.remove_columns(["sentence1","sentence2","idx"])
validation_dataset = validation_dataset.rename_column("label","labels")
validation_dataset.set_format("torch")


Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Load pretrained `bert-base-uncased` with a fresh 2-class classification head (the MLM/NSP head weights are discarded).

In [7]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint,num_labels=2)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Define training configuration — all defaults; checkpoints go to `test-trainer/`.

In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer/checkpoint-1377")

Assemble the `Trainer` with the model, arguments, datasets, collator, and tokeniser.

In [9]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collactor,
    processing_class=tokenizer,
)

Run the training loop.

In [10]:
#trainer.train()

Run inference on the validation set and inspect the output shape.

In [11]:
predictions = trainer.predict(validation_dataset)
print(predictions.predictions.shape, predictions.label_ids.shape)

/home/user/miniconda/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


(408, 2) (408,)


Convert logit outputs to predicted class labels, then compute GLUE MRPC accuracy and F1.

In [12]:
import evaluate
import numpy as np

preds = np.argmax(predictions.predictions, axis=-1)

metric = evaluate.load("glue", "mrpc")
metric.compute(predictions=preds, references=predictions.label_ids)

{'accuracy': 0.8382352941176471, 'f1': 0.888135593220339}